In [1]:
# Explanation: This cell generates all homomorphic images of the 4-uniform tight cycle C_7 with one edge deleted on at most 6 vertices, then excludes those images from the theory.

from collections import defaultdict
from itertools import permutations

target_size = 6
cycle_length = 7
uniformity = 4
source_name = "C7m"


def tight_cycle_edges(n, r=4):
    return [tuple((i + j) % n for j in range(r)) for i in range(n)]


def tight_cycle_minus_edges(n, r=4):
    return tight_cycle_edges(n, r)[:-1]


def canonical_edge_set(num_vertices, edges):
    """Return a canonical representative of an unlabeled 4-graph."""
    best = None
    for perm in permutations(range(num_vertices)):
        relabeled = tuple(sorted({tuple(sorted(perm[v] for v in edge)) for edge in edges}))
        if best is None or relabeled < best:
            best = relabeled
    return best


def generate_homomorphic_images(source_edges, num_source_vertices, max_vertices, r=4):
    """Generate all homomorphic images of the source 4-graph with at most max_vertices vertices, up to isomorphism."""
    images = defaultdict(dict)
    assignment = [None] * num_source_vertices
    assignment[0] = 0

    def complete_edge_is_valid(edge):
        colors = [assignment[v] for v in edge]
        if any(color is None for color in colors):
            return True
        return len(set(colors)) == r

    def partial_assignment_is_valid():
        return all(complete_edge_is_valid(edge) for edge in source_edges)

    def record_image(current_max):
        num_vertices = current_max + 1
        edges = tuple(sorted({tuple(sorted(assignment[v] for v in edge)) for edge in source_edges}))
        key = canonical_edge_set(num_vertices, edges)
        images[num_vertices].setdefault(key, [list(edge) for edge in key])

    def backtrack(pos, current_max):
        if pos == num_source_vertices:
            if partial_assignment_is_valid():
                record_image(current_max)
            return

        upper = min(current_max + 1, max_vertices - 1)
        for color in range(upper + 1):
            assignment[pos] = color
            if partial_assignment_is_valid():
                backtrack(pos + 1, max(current_max, color))
            assignment[pos] = None

    backtrack(1, 0)
    return images


full_cycle_edges = tight_cycle_edges(cycle_length, uniformity)
source_edges = tight_cycle_minus_edges(cycle_length, uniformity)
deleted_edge = full_cycle_edges[-1]

C7m_images_by_size = generate_homomorphic_images(source_edges, cycle_length, target_size, uniformity)
C7m_homomorphic_images = [
    (num_vertices, edges)
    for num_vertices in sorted(C7m_images_by_size)
    for edges in sorted(C7m_images_by_size[num_vertices].values())
]
C7m_image_theories = [
    FourGraphTheory.p(num_vertices, edges=edges)
    for num_vertices, edges in C7m_homomorphic_images
]

edge = FourGraphTheory(4, edges=[[0, 1, 2, 3]])
FourGraphTheory.exclude(C7m_image_theories)
FGT = FourGraphTheory

print(f"Source graph: {source_name} = C_{cycle_length}^(4)-")
print("deleted edge: ", list(deleted_edge))
print("source edge count: ", len(source_edges))
print(f"Homomorphic images of {source_name} with at most {target_size} vertices:")
for num_vertices in range(1, target_size + 1):
    print(f"and size {num_vertices}: ", len(C7m_images_by_size.get(num_vertices, {})))
print("total: ", len(C7m_homomorphic_images))

print("Excluded homomorphic images:")
name_counts = defaultdict(int)
for num_vertices, edges in C7m_homomorphic_images:
    image_index = name_counts[num_vertices]
    name_counts[num_vertices] += 1
    print(f"{source_name}_J{num_vertices}_{image_index} = FourGraphTheory.p({num_vertices}, edges={edges})")

print(f"Number of structures without these {source_name} homomorphic images")
print("and size 5: ", len(FGT.generate(5)))
print("and size 6: ", len(FGT.generate(6)))
# print("and size 7: ", len(FGT.generate(7)))


Source graph: C7m = C_7^(4)-
deleted edge:  [6, 0, 1, 2]
source edge count:  6
Homomorphic images of C7m with at most 6 vertices:
and size 1:  0
and size 2:  0
and size 3:  0
and size 4:  0
and size 5:  0
and size 6:  1
total:  1
Excluded homomorphic images:
C7m_J6_0 = FourGraphTheory.p(6, edges=[[0, 1, 2, 3], [0, 1, 2, 4], [0, 1, 3, 5], [0, 2, 4, 5], [0, 3, 4, 5]])
Number of structures without these C7m homomorphic images
and size 5:  6
and size 6:  80


In [2]:
# Explanation: This cell runs the exact flag algebra SDP for the edge density upper bound. No extremal construction is supplied here.

FGT.optimize(
    edge,
    target_size,
    exact=True,
    file="FourGraph_C7m_6vtx",
    denom=1024*15*3*5,
    slack_threshold=2e-6
)


Base flags generated, their number is 80
The relevant ftypes are constructed, their number is 3
Block sizes before symmetric/asymmetric change is applied: [2, 16, 16]


Done with mult table for Ftype on 4 points with edges=(0123): : 3it [00:01,  1.77it/s]


Tables finished
Constraints finished
Running sdp without construction. Used block sizes are [2, 5, 11, 5, 11, -80, -2]
CSDP 6.2.0
Iter:  0 Ap: 0.00e+00 Pobj:  0.0000000e+00 Ad: 0.00e+00 Dobj:  0.0000000e+00 
Iter:  1 Ap: 1.00e+00 Pobj: -2.7132641e+01 Ad: 6.44e-01 Dobj: -1.4747061e+00 
Iter:  2 Ap: 1.00e+00 Pobj: -2.7247199e+01 Ad: 9.52e-01 Dobj: -4.3025988e-01 
Iter:  3 Ap: 1.00e+00 Pobj: -2.2002133e+01 Ad: 8.96e-01 Dobj: -3.8834093e-01 
Iter:  4 Ap: 1.00e+00 Pobj: -1.5290757e+01 Ad: 6.72e-01 Dobj: -3.7319129e-01 
Iter:  5 Ap: 5.56e-01 Pobj: -1.2032146e+01 Ad: 8.18e-01 Dobj: -3.4546573e-01 
Iter:  6 Ap: 3.90e-01 Pobj: -1.0000724e+01 Ad: 6.45e-01 Dobj: -3.3791531e-01 
Iter:  7 Ap: 9.65e-01 Pobj: -3.1709621e+00 Ad: 8.36e-01 Dobj: -3.3377966e-01 
Iter:  8 Ap: 9.78e-01 Pobj: -8.6063684e-01 Ad: 8.24e-01 Dobj: -3.3801576e-01 
Iter:  9 Ap: 1.00e+00 Pobj: -5.5237209e-01 Ad: 7.51e-01 Dobj: -3.6032389e-01 
Iter: 10 Ap: 1.00e+00 Pobj: -5.0607266e-01 Ad: 7.07e-01 Dobj: -3.8845432e-01 
Iter: 11 Ap:

 33%|████████████████████████▎                                                | 1/3 [00:00<00:00, 17.16it/s]


Rounded X matrix ((5, Ftype on 4 points with edges=(), 6), 1) is not semidefinite: -4.813584103821526e-06
Rounding X matrices


100%|████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 217.63it/s]


Calculating resulting bound


100%|█████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 21.56it/s]

The rounded result is -303349/691200


303349/691200